# Exercise 14 - Transfer Learning using the MNIST Dataset

The goal of this exercise is to implement transfer learning using a simple CNN with the MNIST dataset. The idea is to train the model on the MNIST digits 0-4 then freeze the feature layers of the trained model and re-train just the fully-connected layers of the model using the digits 5-9.

Transfer learning with Keras is well-documented on the web. For example, [Transfer learning & fine-tuning](https://keras.io/guides/transfer_learning/)

- Use **T4 GPU** for this exercise

First run the code below to create and compile the model.

In [ ]:
import time
import tensorflow
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Input
from tensorflow.keras.utils  import to_categorical

image_size = 28
n_labels   = 10

(x_train, y_train), (x_test, y_test) = tensorflow.keras.datasets.mnist.load_data()

# Create two datasets one with digits 0-4 and the other with digits 5-9
x_train_lt5 = x_train[y_train < 5]
y_train_lt5 = y_train[y_train < 5]
x_test_lt5 = x_test[y_test < 5]
y_test_lt5 = y_test[y_test < 5]

x_train_ge5 = x_train[y_train >= 5]
y_train_ge5 = y_train[y_train >= 5]
x_test_ge5 = x_test[y_test >= 5]
y_test_ge5 = y_test[y_test >= 5]

def reshape_and_normalize(x, y):
    x = x.reshape(-1, image_size, image_size, 1).astype('float32')
    x /= 255
    y = to_categorical(y, n_labels)
    return (x, y)

x_train,     y_train     = reshape_and_normalize(x_train, y_train)
x_test,      y_test      = reshape_and_normalize(x_test, y_test)
x_train_lt5, y_train_lt5 = reshape_and_normalize(x_train_lt5, y_train_lt5)
x_test_lt5,  y_test_lt5  = reshape_and_normalize(x_test_lt5, y_test_lt5)
x_train_ge5, y_train_ge5 = reshape_and_normalize(x_train_ge5, y_train_ge5)
x_test_ge5,  y_test_ge5  = reshape_and_normalize(x_test_ge5, y_test_ge5)

# Create a simple CNN model
patch_size     =   3
pool_size      =   2
n_features     =  16
n_full_units   =  32

tensorflow.keras.backend.clear_session()

feature_layers = [
    Input(shape=(image_size,image_size,1)),
    Conv2D(n_features, (patch_size,patch_size), activation='relu'),
    Conv2D(n_features, (patch_size,patch_size), activation='relu'),
    MaxPooling2D(pool_size=(pool_size,pool_size)),
    Conv2D(n_features*2, (patch_size,patch_size), activation='relu'),
    Conv2D(n_features*2, (patch_size,patch_size), activation='relu'),
    MaxPooling2D(pool_size=(pool_size,pool_size)),
    Flatten(),
]

classification_layers = [
    Dense(units=n_full_units, activation='relu'),
    Dense(units=n_labels, activation='softmax'),
]

model = Sequential(feature_layers + classification_layers)
model.summary()
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.save('ex14.keras')

Write your own code to train the model using just the digits 0-4, then evaluate the trained model separately on the two test datasets, the one containing the digits 0-4 and the other containing the digits digits 5-9. Measure how long training takes (you could call `t = time.perf_counter()` before and after training).

In [ ]:
#

Now freeze the feature layers and re-train the model using just the digits 5-9, again evaluating the trained model on both test datasets. You will need to re-compile the model after freezing the feature layers. Compare training times before and after freezing the feature layers.

In [ ]:
#

Finally, with the feature layers still frozen, re-train the model again using the digits 0-9 and evaluate on both test datasets.

In [ ]:
#

#### Solution

Here is our answer. Do not run the cell below unless you want to see the answer we provide

<details> 
    <summary> See our answer </summary>

    def train_model(x, y):
        t = time.perf_counter()
        model.fit(x, y, batch_size=128, epochs=5, shuffle=True)
        print(f'Training time: {time.perf_counter() - t:5.1f} sec')

        loss_and_acc = model.evaluate(x_test_lt5, y_test_lt5, verbose=0)
        print(f'Test on 0-4, accuracy = {100*loss_and_acc[1]:5.1f}%')

        loss_and_acc = model.evaluate(x_test_ge5, y_test_ge5, verbose=0)
        print(f'Test on 5-9, accuracy = {100*loss_and_acc[1]:5.1f}%')

    model = load_model('ex14.keras')

    print('Train on digits 0-4')
    train_model(x_train_lt5, y_train_lt5)

    print('\nFreeze the feature layers')
    for layer in model.layers[:6]:
        layer.trainable = False
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    print('\nTrain on digits 5-9')
    train_model(x_train_ge5, y_train_ge5)

    print('\nTrain on digits 0-9')
    train_model(x_train, y_train)
    
</details>